# House Price Prediction — Linear Regression

## What Are We Building?

We are building a **supervised machine learning model** that predicts house prices based on features like area, bedrooms, age, etc.

### The ML Pipeline

```
Raw Data → Clean Data → Split (Train/Test) → Scale Features → Train Model → Evaluate → Predict
```

| Step | What Happens | Java Analogy |
|---|---|---|
| Load Data | Read CSV into DataFrame | Reading a DB table into a `List<House>` |
| Explore | Check types, nulls, stats | Inspecting your POJO fields and running validation |
| Clean | Fill missing values | Setting default values for null fields |
| Split | Separate train vs test data | Splitting JUnit test data from training data |
| Scale | Normalise feature ranges | Normalising input before processing |
| Train | Model learns patterns | Fitting parameters — like training a spam filter |
| Evaluate | Measure accuracy | Running integration tests and checking metrics |

### What is Linear Regression?

Linear Regression finds the **best straight line** (or hyperplane in multiple dimensions) that fits the data.

```
Price = w1×Area + w2×Bedrooms + w3×Bathrooms + ... + bias
```

- Each **w** (weight/coefficient) tells how much that feature contributes to the price
- The **bias** (intercept) is the base price when all features are zero
- The model learns these values by **minimising the error** between predicted and actual prices

> **Java analogy:** Like finding the best coefficients in a formula — but instead of hand-tuning, the training algorithm (gradient descent) finds them automatically by iterating through the data.

In [2]:
# ============================================================
# STEP 1: Import Libraries
# ============================================================
# pandas       → Data loading and manipulation (like Java's Apache Commons CSV + Collections)
# numpy        → Numerical operations (like Java's Math library on steroids)
# matplotlib   → Plotting charts (like JFreeChart)
# sklearn      → Machine Learning toolkit (no direct Java equivalent — closest is Weka/DL4J)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---
## Step 2: Load and Inspect the Data

Our dataset has **100 houses** with 7 features and 1 target (price).

| Column | Type | Description |
|---|---|---|
| `Area_sqft` | int | Size of the house in square feet |
| `Bedrooms` | int | Number of bedrooms |
| `Bathrooms` | int | Number of bathrooms |
| `Age_years` | int | Age of the house |
| `Distance_to_city_km` | float | Distance to city centre (has some NaN!) |
| `Garage` | int | 1 = has garage, 0 = no garage |
| `Floors` | int | Number of floors |
| **`Price_USD`** | **int** | **Target variable — what we want to predict** |

> **Key ML terms:**
> - **Features (X):** The input columns the model uses to learn (Area, Bedrooms, etc.)
> - **Target (y):** The output column the model tries to predict (Price_USD)
> - **Row = Sample/Observation:** One house in our dataset

In [ ]:
# Load the dataset
df = pd.read_csv('house_price_data.csv')

# Quick overview — how many rows and columns?
print(f'Dataset in tuple : {df.shape}') # it print both row and column
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns') # split to print row in shape[0] and column in shape[1] 
print(f'Features: {df.shape[1] - 1} | Target: 1 (Price_USD)')
print()
df.head() # means shows the first 5 rows of the DataFrame

Dataset shape: 100 rows x 8 columns
Dataset in tuple : (100, 8)
Features: 7 | Target: 1 (Price_USD)



,Area_sqft,Bedrooms,Bathrooms,Age_years,Distance_to_city_km,Garage,Floors,Price_USD
0,3774,2,3,28,16.1,1,3,446385
1,4107,4,2,17,23.3,1,3,489543
2,1460,2,3,17,7.3,0,1,185140
3,1894,2,3,1,NaN,1,1,286522
4,1730,4,2,34,3.5,0,2,212784


In [7]:
# .info() → shows column names, non-null counts, and data types
# KEY INSIGHT: Distance_to_city_km has only 97 non-null values → 3 missing!
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Area_sqft            100 non-null    int64  
 1   Bedrooms             100 non-null    int64  
 2   Bathrooms            100 non-null    int64  
 3   Age_years            100 non-null    int64  
 4   Distance_to_city_km  97 non-null     float64
 5   Garage               100 non-null    int64  
 6   Floors               100 non-null    int64  
 7   Price_USD            100 non-null    int64  
dtypes: float64(1), int64(7)
memory usage: 6.4 KB


In [ ]:
# .describe() → summary statistics for every numeric column
# Look at: min, max (range), mean vs median (50%) for skew, std (spread)
df.describe()
# count row tells you how many rows actually have a value (not null/NaN).
# ex. Distance_to_city_km column have some null value that's why its showing 97. remaining all the column have values.

,Area_sqft,Bedrooms,Bathrooms,Age_years,Distance_to_city_km,Garage,Floors,Price_USD
count,100.000000,100.000000,100.000000,100.000000,97.000000,100.000000,100.000000,100.000000
mean,2717.380000,2.980000,2.040000,21.440000,14.676289,0.420000,1.890000,328377.300000
std,1104.021197,1.476961,0.839913,11.001671,8.517332,0.496045,0.827495,134402.974694
min,621.000000,1.000000,1.000000,0.000000,1.400000,0.000000,1.000000,55418.000000
25%,1807.250000,1.750000,1.000000,14.000000,6.700000,0.000000,1.000000,210127.250000
50%,2791.500000,3.000000,2.000000,22.500000,14.800000,0.000000,2.000000,332383.500000
75%,3622.000000,4.000000,3.000000,31.000000,21.400000,1.000000,3.000000,439099.250000
max,4493.000000,5.000000,3.000000,39.000000,29.700000,1.000000,3.000000,578851.000000


---
## Step 3: Handle Missing Values

ML models **cannot work with NaN** (null) values. We need to handle them before training.

### Common strategies:

| Strategy | When to Use | Code |
|---|---|---|
| **Drop rows** | Very few missing rows, large dataset | `df.dropna()` |
| **Fill with mean** | Numeric column, roughly normal distribution | `df.fillna(df.mean())` |
| **Fill with median** | Numeric column with outliers/skew | `df.fillna(df.median())` |
| **Fill with mode** | Categorical columns | `df.fillna(df.mode()[0])` |
| **Forward fill** | Time series data | `df.fillna(method='ffill')` |

> **We use mean fill** here because Distance_to_city_km is numeric with only 3 missing values, and the distribution looks roughly normal.

In [ ]:
# Check which columns have missing values and how many
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()} out of {df.shape[0] * df.shape[1]} cells")

In [ ]:
# Fill missing Distance_to_city_km with the column mean
df.fillna(df.mean(), inplace=True)

# Verify — no more nulls
print("After cleaning — missing values:")
print(df.isnull().sum())
print(f"\nRow 3 Distance (was NaN, now filled): {df.loc[3, 'Distance_to_city_km']:.2f} km (= mean)")

---
## Step 4: Visualize the Data

Before building a model, we should **look at the data** to understand:
1. **Distribution** of the target variable (Price) — is it normal? skewed?
2. **Correlations** — which features are most related to price?
3. **Relationships** — how does each feature relate to price?

> **Why this matters:** If a feature has zero correlation with price, including it adds noise, not signal. If two features are highly correlated with each other (multicollinearity), the model may become unstable.

In [ ]:
# Distribution of house prices — is the target roughly normal?
plt.figure(figsize=(8, 4))
plt.hist(df['Price_USD'], bins=20, edgecolor='k', color='steelblue')
plt.axvline(df['Price_USD'].mean(), color='red', linestyle='--', label=f'Mean: ${df["Price_USD"].mean():,.0f}')
plt.axvline(df['Price_USD'].median(), color='orange', linestyle='--', label=f'Median: ${df["Price_USD"].median():,.0f}')
plt.title('Distribution of House Prices')
plt.xlabel('Price (USD)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

# If mean ≈ median → roughly symmetric (good for linear regression)
# If mean >> median → right-skewed (may need log transform)

In [ ]:
# Scatter plots — how each feature relates to price visually
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features_list = ['Area_sqft', 'Bedrooms', 'Bathrooms', 'Age_years', 'Distance_to_city_km', 'Garage', 'Floors']

for i, feature in enumerate(features_list):
    ax = axes[i // 4, i % 4]
    ax.scatter(df[feature], df['Price_USD'], alpha=0.5, s=20, color='steelblue')
    ax.set_xlabel(feature)
    ax.set_ylabel('Price')
    ax.set_title(f'{feature} vs Price')

axes[1, 3].axis('off')  # hide empty subplot
plt.suptitle('Feature vs Price Scatter Plots', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — which features correlate most with Price?
plt.figure(figsize=(10, 6))
correlation = df.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# READ THIS: Look at the Price_USD row — features with high absolute correlation
# are the strongest predictors. Positive = price goes up, Negative = price goes down.
print("\nCorrelation with Price_USD (sorted):")
print(correlation['Price_USD'].drop('Price_USD').sort_values(ascending=False).to_string())

---
## Step 5: Split Features (X) and Target (y)

In supervised learning, we always separate:
- **X** = features (input) — what the model sees
- **y** = target (output) — what the model predicts

```
Java analogy:
  X = method parameters (inputs)
  y = return value (output)
  model = the method body that maps inputs → output
```

In [ ]:
# Define features and target
features = ['Area_sqft', 'Bedrooms', 'Bathrooms', 'Age_years', 'Distance_to_city_km', 'Garage', 'Floors']

X = df[features]   # Feature matrix — shape: (100, 7)
y = df['Price_USD'] # Target vector — shape: (100,)

print(f"X shape: {X.shape}  →  {X.shape[0]} houses, {X.shape[1]} features each")
print(f"y shape: {y.shape}  →  {y.shape[0]} price values to predict")
print(f"\nFeatures: {features}")

---
## Step 6: Train/Test Split

**Why split?** We need to test our model on data it has **never seen** during training. This tells us how well it generalises to new houses.

```
100 houses
  ├── 80 houses → Training set (model learns from these)
  └── 20 houses → Test set (model is evaluated on these)
```

| Parameter | Value | Meaning |
|---|---|---|
| `test_size=0.2` | 20% | 20 houses reserved for testing |
| `random_state=42` | seed | Makes the split reproducible (same split every run) |

> **Java analogy:** Like splitting your JUnit test data — you train on one set and validate on a held-out set. `random_state` is like `new Random(42)` — a fixed seed for reproducibility.

### Why not test on the same data we train on?

If you test on training data, you get **artificially high accuracy** — the model has memorised the answers. This is called **overfitting** (like memorising exam answers instead of understanding the material).

In [ ]:
# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} houses")
print(f"Test set:     {X_test.shape[0]} houses")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

---
## Step 7: Feature Scaling (StandardScaler)

**Problem:** Our features have very different ranges:
- `Area_sqft`: 621 – 4,493
- `Bedrooms`: 1 – 5
- `Garage`: 0 – 1

Without scaling, the model gives **disproportionate weight to large-valued features** (Area dominates just because its numbers are bigger).

### StandardScaler formula:

```
scaled_value = (original_value - mean) / standard_deviation
```

After scaling, every feature has **mean = 0** and **std = 1** → all features compete on equal footing.

### Critical rule: Fit on train, transform both

```python
scaler.fit(X_train)           # Learn mean & std from training data ONLY
scaler.transform(X_train)     # Apply to training data
scaler.transform(X_test)      # Apply SAME mean & std to test data
```

> **Why not fit on test data?** That would be **data leakage** — the model indirectly "sees" test data statistics during training, which inflates your evaluation score and gives a false sense of accuracy.

In [ ]:
# Create and fit the scaler on TRAINING data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform in one step
X_test_scaled = scaler.transform(X_test)         # transform only (uses train's mean & std)

# Show before vs after scaling for first row
print("Before scaling (first training sample):")
print(X_train.iloc[0].to_string())
print(f"\nAfter scaling (same sample):")
for name, val in zip(features, X_train_scaled[0]):
    print(f"  {name:25s} → {val:+.3f}")
print(f"\nAll features now on the same scale (mean≈0, std≈1)")

In [ ]:
# Create and train the Linear Regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# The model has now learned these parameters:
print("Learned coefficients (weights for each feature):")
print("-" * 50)
for name, coef in sorted(zip(features, model.coef_), key=lambda x: abs(x[1]), reverse=True):
    direction = "+" if coef > 0 else "-"
    print(f"  {direction} {name:25s}  weight = {coef:>12,.2f}")
print(f"\n  Intercept (bias):            {model.intercept_:>12,.2f}")
print(f"\nRead as: For each 1-unit increase in scaled feature, price changes by the weight amount.")

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("        MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"  MAE  (Mean Absolute Error):  ${mae:>12,.2f}")
print(f"  RMSE (Root Mean Sq Error):   ${rmse:>12,.2f}")
print(f"  R² Score:                     {r2:>11.4f}")
print("=" * 50)
print(f"\n  Interpretation:")
print(f"  → On average, predictions are off by ${mae:,.0f}")
print(f"  → The model explains {r2*100:.1f}% of price variation")

In [ ]:
# Plot 3: Feature Importance — which features matter most?
importance = pd.Series(np.abs(model.coef_), index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importance.plot(kind='barh', color='steelblue', edgecolor='k')
plt.xlabel('Absolute Coefficient (Weight)')
plt.title('Feature Importance — Which Features Drive Price?')
plt.tight_layout()
plt.show()

---
## Summary — What We Learned

### The Complete ML Pipeline:

```
1. LOAD       →  pd.read_csv()
2. EXPLORE    →  .info(), .describe(), .isnull()
3. CLEAN      →  .fillna(mean)
4. VISUALIZE  →  histograms, correlation heatmap, scatter plots
5. SPLIT X/y  →  features vs target
6. SPLIT      →  train_test_split (80/20)
7. SCALE      →  StandardScaler (fit on train, transform both)
8. TRAIN      →  model.fit(X_train, y_train)
9. PREDICT    →  model.predict(X_test)
10. EVALUATE  →  MAE, RMSE, R²
```

### Key Terms to Remember:

| Term | Definition |
|---|---|
| **Feature** | Input variable (X) — what the model uses to predict |
| **Target** | Output variable (y) — what the model predicts |
| **Training set** | Data the model learns from (80%) |
| **Test set** | Data the model is evaluated on (20%) — never seen during training |
| **Overfitting** | Model memorises training data, fails on new data |
| **Data leakage** | Accidentally using test data info during training |
| **Coefficient** | Weight learned for each feature |
| **Intercept** | Base prediction when all features = 0 |
| **R² Score** | % of target variance explained by the model (0 to 1) |
| **MAE** | Average absolute prediction error in dollars |
| **RMSE** | Like MAE but penalises big errors more |
| **StandardScaler** | Normalises features to mean=0, std=1 |

### What Could Improve This Model?

1. **More data** — 100 rows is very small; real datasets have thousands+
2. **Feature engineering** — create new features like `price_per_sqft`, `bedrooms_per_floor`
3. **Try other algorithms** — Decision Trees, Random Forest, Gradient Boosting
4. **Cross-validation** — instead of one 80/20 split, use k-fold CV for more robust evaluation
5. **Hyperparameter tuning** — optimise model settings with GridSearchCV

In [ ]:
# Side-by-side comparison of actual vs predicted for all test houses
results = pd.DataFrame({
    'Actual Price': y_test.values,
    'Predicted Price': y_pred.round(0),
    'Error': (y_test.values - y_pred).round(0),
    'Error %': ((np.abs(y_test.values - y_pred) / y_test.values) * 100).round(1)
})
results.index = range(1, len(results) + 1)
results.index.name = 'House #'

print(results.to_string())
print(f"\nAverage error: {results['Error %'].mean():.1f}%")

---
## Step 12: Compare Predictions vs Actual (Test Set Table)

In [ ]:
# Predict price for a NEW house
new_house = pd.DataFrame({
    'Area_sqft': [2500],
    'Bedrooms': [3],
    'Bathrooms': [2],
    'Age_years': [10],
    'Distance_to_city_km': [8.0],
    'Garage': [1],
    'Floors': [2]
})

# IMPORTANT: Must scale the new data using the SAME scaler (fitted on training data)
new_house_scaled = scaler.transform(new_house)
predicted_price = model.predict(new_house_scaled)[0]

print("New House Details:")
print("-" * 40)
for col in features:
    print(f"  {col:25s}: {new_house[col].values[0]}")
print("-" * 40)
print(f"  Predicted Price:         ${predicted_price:,.2f}")

---
## Step 11: Predict on a New House

This is the whole point — use the trained model to predict the price of a **brand new house** that wasn't in the dataset.

In [ ]:
# Plot 2: Residuals (errors) — should be randomly scattered around zero
residuals = y_test.values - y_pred

plt.figure(figsize=(8, 4))
plt.scatter(y_pred, residuals, alpha=0.7, color='coral', edgecolors='k', s=60)
plt.axhline(y=0, color='black', linestyle='--', lw=1)
plt.xlabel('Predicted Price (USD)')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residual Plot — Are Errors Random?')
plt.tight_layout()
plt.show()

# GOOD: residuals randomly scattered around 0 → model captures the pattern well
# BAD: residuals show a curve/pattern → linear regression is missing something

In [ ]:
# Plot 1: Actual vs Predicted — the closer to the diagonal, the better
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='steelblue', edgecolors='k', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction line')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.tight_layout()
plt.show()

# Points ON the red line = perfect prediction
# Points ABOVE the line = model over-predicted
# Points BELOW the line = model under-predicted

---
## Step 10: Visualize Results

---
## Step 9: Make Predictions and Evaluate

Now we use the trained model to predict prices for the **test set** (20 houses it has never seen).

### Evaluation Metrics Explained

| Metric | Formula | What It Tells You | Good Value |
|---|---|---|---|
| **MAE** (Mean Absolute Error) | avg(\|actual - predicted\|) | Average dollar amount the prediction is off | Lower = better |
| **RMSE** (Root Mean Squared Error) | sqrt(avg((actual - predicted)²)) | Like MAE but penalises large errors more | Lower = better |
| **R² Score** | 1 - (sum of squared errors / total variance) | % of price variation explained by the model | Closer to 1.0 = better |

### How to interpret R²:
- **R² = 1.0** → Perfect predictions (never happens in real life)
- **R² = 0.8** → Model explains 80% of price variation (good)
- **R² = 0.5** → Model explains 50% (mediocre)
- **R² = 0.0** → Model is no better than just guessing the average price
- **R² < 0** → Model is worse than guessing the average (something went wrong)

---
## Step 8: Train the Model

Now we feed the scaled training data into `LinearRegression` and let it learn the weights.

### What happens inside `.fit()`?

The algorithm uses **Ordinary Least Squares (OLS)** to find weights that minimise:

```
Total Error = sum of (actual_price - predicted_price)² for all training houses
```

It solves this mathematically in one step (no iterations needed for linear regression).

> **Java analogy:** `.fit()` is like calling `model.train(trainingData)` — after this call, the model object holds the learned weights internally.